<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Read-Data" data-toc-modified-id="Read-Data-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Read Data</a></span></li></ul></div>

# Read Data

In [16]:
import json
import pandas as pd

# Open the JSON file and read it as a dictionary
with open('.\json_outputs\pglib_opf_case30_ieee.json', 'r') as file:
    mpc_data_json = json.load(file)

print(Datakeys:=mpc_data_json.keys())


dict_keys(['baseMVA', 'bus', 'gen', 'gencost', 'branch'])


<>:5: SyntaxWarning: invalid escape sequence '\j'
<>:5: SyntaxWarning: invalid escape sequence '\j'
C:\Users\Syed\AppData\Local\Temp\ipykernel_18532\2411087010.py:5: SyntaxWarning: invalid escape sequence '\j'
  with open('.\json_outputs\pglib_opf_case30_ieee.json', 'r') as file:


In [17]:
mpc_data = {}
for k in Datakeys:
    mpc_data[k] = pd.DataFrame(mpc_data_json[k])

In [20]:
mpc_data['gencost']

,bus_i,gen_ID,MODEL,STARTUP,SHUTDOWN,NCOST,COST_2,COST_1,COST_0
0,1,1,2,0,0,3,0,32.112559,0
1,2,2,2,0,0,3,0,61.798693,0
2,5,3,2,0,0,3,0,0.000000,0
3,8,4,2,0,0,3,0,0.000000,0
4,11,5,2,0,0,3,0,0.000000,0
5,13,6,2,0,0,3,0,0.000000,0


In [19]:
import pyomo.environ as pye
import numpy as np

# Create pyomo Model
model = pye.ConcreteModel()

# === Sets ===
model.BUSES = pye.Set(initialize=mpc_data['bus'].index.tolist())  # Example subset of buses
model.GENS = pye.Set(initialize=mpc_data["gen"]['gen_ID'].tolist())  # Active generators
model.BRANCHES = pye.Set(initialize=mpc_data['branch'].index.tolist())  # Transmission lines


# === Parameters ===
# Generator cost coefficients (only linear term because c2=0)
if np.all( 
    (mpc_data['gencost']['c2']==0) & (mpc_data['gencost']['c0']==0)
    ):
        model.c1 = pye.Param(model.GENS, 
                             initialize= dict(zip(
                                        mpc_data['gencost']["gen_ID"], mpc_data['gencost']['c1']
                                            ))
                            )  # Linear cost ($/MW)
else:
    print('handle nonlinear costs')
    
# Bus power demand (MW)
model.Pd = pye.Param(model.BUSES, initialize=mpc_data['bus']['Pd'].to_dict())

# Generator capacity limits (MW)
model.Pmax = pye.Param(model.GENS, initialize=dict(zip(
                                        mpc_data['gen']["gen_ID"], mpc_data['gen']['Pmax']
                                            ))
                      )
model.Pmin = pye.Param(model.GENS, initialize=dict(zip(
                                        mpc_data['gen']["gen_ID"], mpc_data['gen']['Pmin']
                                            ))
                      )

# Transmission line limits (MW)
model.Pmax_line = pye.Param(model.BRANCHES, initialize=mpc_data['branch']['rateA'].to_dict())

# Line susceptance (1/X), assuming per unit values
model.B = pye.Param(model.BRANCHES, initialize=
                    (
                        mpc_data['branch']['x']/(mpc_data['branch']['x']**2+mpc_data['branch']['r']**2)
                    ).to_dict())

# Generator bus assignment
model.gen_bus = pye.Param(model.GENS, initialize=dict(zip(
                        mpc_data['gen']['gen_ID'].values, mpc_data['gen']['gen_ID'].index
                                                        ))
                         )

# === Variables ===
model.Pg = pye.Var(model.GENS, within=pye.NonNegativeReals, 
                   bounds=lambda model, g: (model.Pmin[g], model.Pmax[g])
                  )  # Generator output
model.theta = pye.Var(model.BUSES, within=pye.Reals,
                     bounds=( -100,  100))  # Voltage angle (radians)np.radians(30)
model.P_flow = pye.Var(model.BRANCHES, within=pye.Reals, 
                       bounds=lambda model, fbus, tbus: (-model.Pmax_line[fbus, tbus], model.Pmax_line[fbus, tbus])
                      )  # Line flows
# model.slack = pyo.Var(model.BRANCHES, within=pyo.NonNegativeReals)



# === Objective Function (Minimize Generation Cost) ===
model.objective = pye.Objective(rule=sum(model.c1[g] * model.Pg[g] for g in model.GENS), 
                                sense=pye.minimize)

# === Power Balance Constraints ===
model.power_balance = pye.ConstraintList()
for b in model.BUSES:
    model.power_balance.add(
        sum(model.Pg[g] for g in model.GENS if model.gen_bus[g] == b) 
        + sum(model.P_flow[l] for l in model.BRANCHES if l[1] == b) 
        - sum(model.P_flow[l] for l in model.BRANCHES if l[0] == b) 
        == model.Pd[b]
    )
    
    
# === Line Flow Constraints (DC Power Flow) ===
model.line_flow = pye.ConstraintList()
for l in model.BRANCHES:
    model.line_flow.add(
        model.P_flow[l] == model.B[l]  * (model.theta[l[0]] - model.theta[l[1]])   #+ model.slack[l]
    )


# === Reference Bus Constraint (Slack Bus) ===
model.theta_ref = pye.Constraint(expr=model.theta[1] == 0)  # Bus 1 is the reference bus


# === Solve Model Using Gurobi ===
solver = pye.SolverFactory("gurobi")  # Use Gurobi solver
solver.options["OptimalityTol"] = 1e-8  # Higher precision
solver.solve(model, tee=True)

# === Display Results ===
print("\nOptimal Generator Outputs (MW):")
for g in model.GENS:
    print(f"Generator {g}: {pye.value(model.Pg[g])} MW")

print("\nOptimal Line Flows (MW):")
for n, l in enumerate(model.BRANCHES,1):
    print(f"Line {n}:={l}: {pye.value(model.P_flow[l])} MW")

print("\nVoltage Angles (Radians):")
for b in model.BUSES:
    print(f"Bus {b}: {pye.value(model.theta[b])} rad")


KeyError: 'c2'